# Contextual Adaptive Stacking — эксперименты в Google Colab

Этот ноутбук:
1. Монтирует Google Drive (результаты переживают перезапуск сессии)
2. Ставит зависимости и клонирует `ranker_lab`
3. Запускает base-модели и adaptive stacking
4. Сохраняет `runs.parquet`, кеш и артефакты на Drive

**Рекомендуемый runtime:** GPU (T4 достаточно). CatBoost работает на CPU, нейросети — на GPU.

## 1. Google Drive и пути

Структура на Drive (создаётся автоматически):
```
MyDrive/diploma/
  data/raw/books5_interactions.parquet   ← сырые данные
  cache/                                  ← rank_table (переиспользуется)
  results/                                ← runs.parquet + артефакты
  ranker_lab/                             ← код (git clone)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ── Настройте под себя ──────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/diploma')  # корень проекта на Drive
GIT_BRANCH = 'ttb_novelty'                           # ветка с stacking
GIT_REPO   = 'https://github.com/gl-egor/ranker_lab.git'

# Пути, которые переживают перезапуск Colab
RAW_DIR    = DRIVE_ROOT / 'data' / 'raw'
CACHE_DIR  = DRIVE_ROOT / 'cache'
RESULTS_DIR = DRIVE_ROOT / 'results'
REPO_DIR   = DRIVE_ROOT / 'ranker_lab'

for p in [RAW_DIR, CACHE_DIR, RESULTS_DIR, DRIVE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Raw data:  ', RAW_DIR)
print('Cache:     ', CACHE_DIR)
print('Results:   ', RESULTS_DIR)

## 2. Установка зависимостей и код проекта

In [ ]:
!pip install -q catboost torch pandas pyarrow scikit-learn tqdm

In [ ]:
import os
import subprocess
import sys

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', GIT_BRANCH], check=True)
    print(f'Updated {REPO_DIR}')
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, GIT_REPO, str(REPO_DIR)], check=True)
    print(f'Cloned to {REPO_DIR}')

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

# loader ищет сырые данные здесь
os.environ['RLAB_RAW_DIR'] = str(RAW_DIR)

print('CWD:', os.getcwd())

## 2b. Проверка версии кода (обязательно перед экспериментами)

Если видите `ValueError: All arrays must be of the same length` после CatBoost — у вас **старый код в памяти Colab**. Выполните эту ячейку и при необходимости **Runtime → Restart session**.

In [ ]:
import importlib
import inspect
import shutil

# Сброс закешированных модулей (важно после git pull!)
for mod in list(sys.modules):
    if mod.startswith('rlab'):
        del sys.modules[mod]

import rlab.stacking.base_scores as base_scores
importlib.reload(base_scores)

from rlab.stacking import check_stacking_cache_fix

print('base_scores file:', inspect.getfile(base_scores))
print('CACHE_VERSION:', base_scores.STACKING_CACHE_VERSION)
print('check_stacking_cache_fix():', check_stacking_cache_fix())

assert check_stacking_cache_fix(), (
    'Старая версия base_scores! Выполните git pull и Runtime → Restart session.'
)

# Удалить старый битый кеш stacking (формат v1: один .parquet)
stacking_dir = RESULTS_DIR / 'stacking'
if stacking_dir.exists():
    removed = 0
    for legacy in stacking_dir.rglob('*.parquet'):
        if legacy.name in ('train.parquet', 'valid.parquet', 'test.parquet'):
            continue  # v2 — ok, но v3 использует scores.npz
        legacy.unlink(missing_ok=True)
        removed += 1
    for d in stacking_dir.rglob('*__oof*'):
        if d.is_dir() and not (d / 'scores.npz').exists():
            shutil.rmtree(d, ignore_errors=True)
            removed += 1
    print(f'Cleaned legacy stacking cache entries: {removed}')

print('OK — можно запускать эксперименты')

## 3. Сырые данные (Amazon Books 5-core)

Положите `books5_interactions.parquet` на Drive **один раз**:
`MyDrive/diploma/data/raw/books5_interactions.parquet`

Колонки: `user_id`, `item_id`, `rating`, `timestamp`.

Если файла ещё нет — загрузите с локального компьютера:

In [ ]:
from google.colab import files

RAW_FILE = RAW_DIR / 'books5_interactions.parquet'

if RAW_FILE.exists():
    print(f'OK: {RAW_FILE} ({RAW_FILE.stat().st_size / 1e6:.1f} MB)')
else:
    print('Файл не найден. Загрузите parquet (может занять несколько минут)...')
    uploaded = files.upload()  # выберите books5_interactions.parquet
    for name, data in uploaded.items():
        dest = RAW_DIR / name
        dest.write_bytes(data)
        print(f'Saved to {dest}')

## 4. Базовый конфиг (все пути → Drive)

`output_dir` и `cache_dir` на Drive — результаты не пропадут после disconnect.

In [ ]:
import torch
from rlab.configs import DataConfig, EvalConfig, ExperimentConfig, ModelConfig, StackingConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

BASE_MODEL_PARAMS = {
    'catboost': {},
    'dcnv2_enhanced': {
        'max_epochs': 20, 'patience': 3, 'groups_per_batch': 256,
        'lr': 1e-3, 'hard_mining': False,
    },
    'finalmlp': {
        'max_epochs': 20, 'patience': 3, 'groups_per_batch': 256,
        'lr': 1e-3, 'tail_aware_alpha': 0.3, 'loss_temperature': 0.7,
    },
}

def make_stacking_cfg(
    name: str,
    *,
    train_size: int = 30_000,
    seed: int = 42,
    n_oof_folds: int = 5,
    calibrate: bool = True,
    bootstrap_n: int = 1000,
    max_users: int | None = None,
) -> ExperimentConfig:
    return ExperimentConfig(
        name=name,
        data=DataConfig(
            dataset='books5',
            train_size=train_size,
            valid_size=10_000,
            test_size=10_000,
            feature_set='full',
            max_users=max_users,
        ),
        model=ModelConfig(kind='adaptive_stacking'),
        eval=EvalConfig(
            k=10, stratify_by_pop=True, n_pop_bins=4,
            bootstrap_n=bootstrap_n,
        ),
        stacking=StackingConfig(
            base_models=['catboost', 'dcnv2_enhanced', 'finalmlp'],
            base_model_params=BASE_MODEL_PARAMS,
            n_oof_folds=n_oof_folds,
            calibrate=calibrate,
        ),
        seed=seed,
        output_dir=str(RESULTS_DIR),   # ← Google Drive
        cache_dir=str(CACHE_DIR),     # ← Google Drive
        device=DEVICE,
    )

## 5. Запуск экспериментов

### 5a. Быстрая проверка (~20–40 мин на GPU)

Малый train, 3 OOF folds. Проверяет, что pipeline работает.

In [ ]:
from rlab.stacking import run_stacking_experiment

cfg_quick = make_stacking_cfg(
    'stacking_colab_quick',
    train_size=3_000,
    n_oof_folds=3,
    bootstrap_n=0,
    max_users=2_000,
)
cfg_quick.stacking.gate_epochs = 20
cfg_quick.stacking.base_model_params = {
    'catboost': {'iterations': 100},
    'dcnv2_enhanced': {**BASE_MODEL_PARAMS['dcnv2_enhanced'], 'max_epochs': 5},
    'finalmlp': {**BASE_MODEL_PARAMS['finalmlp'], 'max_epochs': 5},
}

record = run_stacking_experiment(cfg_quick)
print(record.summary())

### 5b. Основной эксперимент для диплома (3 seeds)

**Время:** ~6–10 ч на GPU T4. Можно запускать по одному seed за сессию — результаты дописываются в `runs.parquet` (skip_if_exists=True).

In [ ]:
records_main = []
for seed in [42, 43, 44]:
    cfg = make_stacking_cfg(f'stacking_colab_main_s{seed}', train_size=30_000, seed=seed)
    rec = run_stacking_experiment(cfg)  # пропустит, если run_id уже есть
    records_main.append(rec)
    print(rec.summary())
    print()

### 5c. Сегментный анализ (новизна диплома)

Warm users + все сегменты в `stacking_extras.json`.

In [ ]:
cfg_seg = make_stacking_cfg('stacking_colab_segments', train_size=30_000)
cfg_seg.eval.eval_warm_only = True

rec_seg = run_stacking_experiment(cfg_seg)
print(rec.summary())

### 5d. Ablation: с calibration / без

Сравнение для раздела «методология».

In [ ]:
for name, calibrate in [('stacking_colab_cal_yes', True), ('stacking_colab_cal_no', False)]:
    cfg = make_stacking_cfg(name, calibrate=calibrate)
    rec = run_stacking_experiment(cfg)
    print(rec.summary())
    print()

### 5e. Одиночные base-модели (для таблицы сравнения)

Если ещё не гоняли CatBoost / DCN-v2 / FinalMLP отдельно.

In [ ]:
from rlab.runner import run_experiment

for kind in ['catboost', 'dcnv2_enhanced', 'finalmlp']:
    cfg = ExperimentConfig(
        name=f'base_{kind}_colab',
        data=DataConfig(dataset='books5', train_size=30_000, feature_set='full'),
        model=ModelConfig(kind=kind, params=BASE_MODEL_PARAMS.get(kind, {})),
        eval=EvalConfig(k=10, stratify_by_pop=True, bootstrap_n=1000),
        seed=42,
        output_dir=str(RESULTS_DIR),
        cache_dir=str(CACHE_DIR),
        device=DEVICE,
    )
    rec = run_experiment(cfg)
    print(rec.summary())
    print()

## 6. Просмотр и выгрузка результатов

Всё уже на Drive. Ниже — сводная таблица и скачивание на локальный ПК.

In [ ]:
import json
import pandas as pd

RUNS_PATH = RESULTS_DIR / 'runs.parquet'

if RUNS_PATH.exists():
    df = pd.read_parquet(RUNS_PATH)
    cols = ['run_id', 'model', 'seed', 'train_size', 'ndcg_at_k', 'hr_at_k',
            'ndcg_ci_low', 'ndcg_ci_high', 'ndcg_tail', 'ndcg_head']
    cols = [c for c in cols if c in df.columns]
    display(df[cols].sort_values('ndcg_at_k', ascending=False).tail(20))
else:
    print('runs.parquet ещё не создан — сначала запустите эксперимент')

In [ ]:
# Детали stacking: сегменты и веса gating
stacking_runs = df[df['model'] == 'adaptive_stacking'] if RUNS_PATH.exists() else []

if len(stacking_runs):
    last_run_id = stacking_runs.iloc[-1]['run_id']
    extras_path = RESULTS_DIR / 'runs' / last_run_id / 'stacking_extras.json'
    if extras_path.exists():
        with open(extras_path) as f:
            extras = json.load(f)
        print('=== NDCG по сегментам ===')
        for seg_name, seg_vals in extras.get('segments', {}).items():
            print(f'\n{seg_name}:')
            for k, v in seg_vals.items():
                print(f'  {k}: {v:.4f}')
        print('\n=== Gating weights по popularity ===')
        for seg, weights in extras.get('gating_weights_by_pop_bin', {}).items():
            print(f'  {seg}: {weights}')
        print('\n=== Baselines ===')
        print('Equal blend:', extras.get('baseline_equal_blend', {}))
        for k, v in extras.get('baseline_base_models', {}).items():
            print(f'  {k}: NDCG={v.get("NDCG", 0):.4f}')

In [ ]:
# Скачать runs.parquet на локальный компьютер
from google.colab import files

if RUNS_PATH.exists():
    files.download(str(RUNS_PATH))
else:
    print('Нечего скачивать')

## 7. Что где лежит на Google Drive

| Путь на Drive | Содержимое | Зачем |
|---|---|---|
| `diploma/data/raw/` | `books5_interactions.parquet` | Сырые данные (загрузить один раз) |
| `diploma/cache/` | Подготовленные rank_table | Ускоряет повторные запуски |
| `diploma/results/runs.parquet` | Все эксперименты | Главная таблица для диплома |
| `diploma/results/runs/<run_id>/` | config.json, stacking_extras.json, gating_model.pt | Детали каждого запуска |
| `diploma/results/stacking/` | Кеш OOF-скоров base-моделей | Не переобучать base при ablation |

### Советы для Colab

1. **Runtime disconnect** — результаты на Drive сохраняются; при повторном запуске `skip_if_exists=True` пропустит готовые run_id.
2. **Один seed за сессию** — для main эксперимента запускайте цикл по seeds по одному, если не хватает времени.
3. **Сначала quick** — убедитесь, что данные и GPU работают, потом main.
4. **Base scores кеш** — после первого stacking повторные ablation/segments не переобучают base (если hash конфига совпадает).
5. **Скачать всё:** в Drive можно zip-папку `diploma/results/` и скачать через веб-интерфейс Drive.

In [ ]:
# Опционально: zip results для скачивания одним файлом
import shutil

zip_path = '/content/diploma_results'
shutil.make_archive(zip_path, 'zip', RESULTS_DIR)
files.download(f'{zip_path}.zip')